In [1]:
import torch
from torch import nn
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
import torch_directml

In [2]:
BATCH_SIZE = 128
NUM_WORKERS = 0
LEARNING_RATE = 1e-3
DEVICE = torch_directml.device(0)
EPOCHS = 25

In [3]:
transform = transforms.Compose([
    transforms.Resize((150, 150)), 
    transforms.ToTensor()
])

In [4]:
train_data = ImageFolder(root='data/intel-image/train', transform=transform)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=False, num_workers=0)

test_data = ImageFolder(root='data/intel-image/test', transform=transform)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=False, num_workers=0)

In [5]:
train_data[0][0].shape

torch.Size([3, 150, 150])

In [6]:
train_data.class_to_idx

{'buildings': 0,
 'forest': 1,
 'glacier': 2,
 'mountain': 3,
 'sea': 4,
 'street': 5}

In [7]:
class GenerativeVariationalAutoencoder(nn.Module):
    def __init__(self, latent_dim=1024, num_classes=len(train_data.classes)):
        super().__init__()
        self.num_classes = num_classes

        # Encoder
        self.encoder = nn.Sequential(
            # [3, 150, 150] -> [16, 75, 75]
            nn.Conv2d(in_channels=3+num_classes, out_channels=16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(num_features=16),
            nn.ReLU(inplace=True),
            # [16, 75, 75] -> [64, 38, 38]
            nn.Conv2d(in_channels=16, out_channels=64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True),
            # [64, 38, 38] -> [128, 19, 19]
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(num_features=128),
            nn.ReLU(inplace=True)
        )

        self.flatten_size = 128 * 19 * 19

        # Bottleneck
        self.fc_mu = nn.Linear(self.flatten_size, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_size, latent_dim)

        # Decoder
        self.decoder_input = nn.Linear(latent_dim + num_classes, self.flatten_size)

        self.decoder = nn.Sequential(
            # [128, 19, 19] -> [64, 38, 38]
            nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True),
            # [64, 38, 38] -> [16, 75, 75]
            nn.ConvTranspose2d(in_channels=64, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=0),
            nn.BatchNorm2d(num_features=16),
            nn.ReLU(inplace=True),
            # [16, 75, 75] -> [3, 150, 150]
            nn.ConvTranspose2d(in_channels=16, out_channels=3, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def encode(self, x, c):
        batch_size = x.size(0)

        c_spatial = c.view(batch_size, self.num_classes, 1, 1).expand(-1, -1, 150, 150)
        x_cond = torch.cat((x, c_spatial), dim=1)
        h = self.encoder(x_cond)
        h = h.view(batch_size, -1)

        return self.fc_mu(h), self.fc_logvar(h)

    def reparametrize(self, mu, logvar):
        logvar = torch.clamp(logvar, min=-10, max=10)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        return mu + eps * std

    def decode(self, z, c):
        z_cond = torch.cat([z, c], dim=1)
        h = self.decoder_input(z_cond)
        h = h.view(h.size(0), 128, 19, 19)

        return self.decoder(h)

    def forward(self, x, c):
        mu, logvar = self.encode(x, c)
        logvar = torch.clamp(logvar, min=-10, max=10)
        z = self.reparametrize(mu, logvar)
        reconstructed = self.decode(z, c)

        return reconstructed, mu, logvar

In [8]:
def vae_loss_function(reconstructed, x, mu, logvar, beta=0.01):
    MSE = nn.functional.mse_loss(reconstructed, x, reduction='mean')
    KLD = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    return MSE + beta * KLD

In [9]:
model = GenerativeVariationalAutoencoder()
model.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

In [ ]:
import gc

model.train()
for epoch in range(EPOCHS):
    train_loss = 0.0
    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels_one_hot = nn.functional.one_hot(labels, num_classes=len(train_data.classes)).float().to(DEVICE)
        reconstructed, mu, logvar = model(images, labels_one_hot)
        loss = vae_loss_function(reconstructed, images, mu, logvar)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        train_loss += loss.item()


    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            labels_one_hot = nn.functional.one_hot(labels, num_classes=len(train_data.classes)).float().to(DEVICE)
                
            reconstructed, mu, logvar = model(images, labels_one_hot)
            loss = vae_loss_function(reconstructed, images, mu, logvar)
            val_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(test_loader):.4f}")


    scheduler.step()
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
import matplotlib.pyplot as plt

def target_generation(model, num_samples, target_class):
    model.eval()

    with torch.no_grad():
        labels = torch.full((num_samples,), target_class, dtype=torch.long)
        labels_one_hot = nn.functional.one_hot(labels, num_classes=len(train_data.classes)).float().to(DEVICE)

        z_random = torch.randn(num_samples, 1024).to(DEVICE)

        generated_images = model.decode(z_random, labels_one_hot)
        generated_images = generated_images.cpu().numpy()

    fig, axes = plt.subplots(1, num_samples, figsize=(num_samples * 2, 2))
    for i in range(num_samples):
        axes[i].imshow(generated_images[i].transpose(1, 2, 0))
        axes[i].axis('off')
        axes[i].set_title(f"Class: {train_data.classes[target_class]}")

    plt.tight_layout()
    plt.show()


In [ ]:
target_generation(model=model, num_samples=5, target_class=0)  # Change target_class to generate images for different classes